# Phase 0 — 기존 로그 재분석 (새 학습 0분)

Kaggle 원본 산출물은 소실됐지만(`docs/PROGRESS.md` §8), 아카이브 노트북의 **출력 로그**에서
25개 run 의 에폭별 지표를 복구했다. 그것만으로 답할 수 있는 두 가지를 본다.

1. **arm 내부 궤적** — decay 구간에서 sharpness 와 accuracy 가 같이 움직이는가.
   raw 상관은 둘 다 시간에 단조라 생기는 허위 상관이므로 **1차 차분**으로 판단한다.
2. **across-run 산점도** — 측정 프로토콜이 3세대로 갈려 비교 가능한 Gen 2 만 쓴다.
   베이스라인 3종은 sharpness 측정 자체가 없다.

**GPU 불필요.** 로컬에서도 돈다.

In [ ]:
# ══ 1. 준비 ═══════════════════════════════════════════════════════════════
import os, csv, json, glob, math

# 리포 루트로 이동 (results/parsed 를 읽는다)
while not os.path.isdir('results/parsed') and os.getcwd() != '/':
    os.chdir('..')
print('작업 경로:', os.getcwd())

PARSED, OUTDIR = 'results/parsed', 'results/phase0'
os.makedirs(OUTDIR, exist_ok=True)

def mean(v): return sum(v)/len(v) if v else float('nan')
def pearson(x, y):
    n = len(x)
    if n < 3: return float('nan')
    mx, my = mean(x), mean(y)
    sx = sum((a-mx)**2 for a in x)**.5; sy = sum((b-my)**2 for b in y)**.5
    if sx == 0 or sy == 0: return float('nan')
    return sum((a-mx)*(b-my) for a,b in zip(x,y))/(sx*sy)
def ranks(v):
    r = [0]*len(v)
    for pos, i in enumerate(sorted(range(len(v)), key=lambda i: v[i])): r[i] = pos+1
    return r
def spearman(x, y): return pearson(ranks(x), ranks(y))
def diff(v): return [v[i+1]-v[i] for i in range(len(v)-1)]

def read_csv(p):
    with open(p) as f: return list(csv.DictReader(f))
def load(fn):
    p = os.path.join(PARSED, fn)
    return read_csv(p) if os.path.exists(p) else None

print('parsed run:', len(glob.glob(os.path.join(PARSED, '*.csv'))) - 1)

In [ ]:
# ══ 2. (필요시) 노트북 로그 재파싱 ════════════════════════════════════════
# 아카이브 노트북을 수정했거나 results/parsed 가 비었을 때만 실행하면 된다.
!python3 scripts/parse_notebook_logs.py

In [ ]:
# ══ 3. arm 내부 궤적 — 핵심 분석 ══════════════════════════════════════════
# decay 구간 안에서 sharpness 와 test acc 가 같이 움직이는가?
#
# 주의: raw 상관을 그대로 믿으면 안 된다. sam_on 에서는 sharpness 와 accuracy 가
# 둘 다 에폭에 대해 단조라, 상관이 −0.9 를 넘어도 그건 "둘 다 시간에 단조"라는
# 사실의 반영일 뿐이다. 1차 차분으로 추세를 제거한 Δ 로 판단한다.
DECAY_ARMS = [
    ('ResNet v1 decay10 sam_off', '09_wsd_sam_overlay_v1__c8.csv'),
    ('ResNet v1 decay10 sam_on',  '09_wsd_sam_overlay_v1__c9.csv'),
    ('ResNet v2 decay20 sam_off', '10_wsd_sam_overlay_v2__c8.csv'),
    ('ResNet v2 decay20 sam_on',  '10_wsd_sam_overlay_v2__c9.csv'),
    ('ViT decay10 sam_off',       '11_tiny_imagenet_wsd_sam__c7.csv'),
    ('ViT decay10 sam_on',        '11_tiny_imagenet_wsd_sam__c8.csv'),
]

traj = []
print(f'{"arm":<28}{"n":>3}{"raw":>8}{"rank":>8}{"e~s":>8}{"e~a":>8}{"Δ":>9}')
print('-'*72)
for name, fn in DECAY_ARMS:
    rows = load(fn)
    if not rows: print(f'{name:<28}  (파일 없음)'); continue
    rows = [r for r in rows if r['sharpness'] and r['test_acc']]
    if len(rows) < 3: print(f'{name:<28}  (표본 부족)'); continue
    e = [float(r['epoch']) for r in rows]
    s = [float(r['sharpness']) for r in rows]
    a = [float(r['test_acc']) for r in rows]
    rec = dict(arm=name, n=len(s), raw=pearson(s,a), rank=spearman(s,a),
               epoch_sharp=pearson(e,s), epoch_acc=pearson(e,a),
               detrended=pearson(diff(s), diff(a)),
               acc_start=a[0], acc_end=a[-1], sharp_start=s[0], sharp_end=s[-1])
    traj.append(rec)
    print(f'{name:<28}{rec["n"]:>3}{rec["raw"]:>8.3f}{rec["rank"]:>8.3f}'
          f'{rec["epoch_sharp"]:>8.3f}{rec["epoch_acc"]:>8.3f}{rec["detrended"]:>9.3f}')
print('-'*72)
print()
print('해석: e~a 가 +0.9 이상이면 accuracy 는 에폭에 대해 사실상 단조 상승이다.')
print('      sam_on 은 e~s 도 −0.9 근처라 raw 상관이 커 보이는 것이고,')
print('      Δ 의 부호가 arm 마다 뒤집히면 "추세를 빼면 무관"이 결론이다.')

In [ ]:
# ══ 4. across-run 산점도 (보조) ═══════════════════════════════════════════
# sharpness 측정 프로토콜이 3세대로 갈려 있어 전부를 한 축에 올릴 수 없다.
GEN = {'01': 'none', '02': 'gen1', '03': 'gen1', '04': 'gen0', '05': 'gen0',
       '06': 'gen0', '07': 'gen1', '08': 'gen2', '09': 'gen2', '10': 'gen2',
       '11': 'vit'}
LABEL = {'gen2': '✅ Gen 2 (ResNet, 고정 probe) — 비교 가능',
         'vit':  '△  Gen 2 (ViT) — loss·해상도가 달라 별도 축',
         'gen1': '❌ Gen 1 — probe 가 run 마다 다름',
         'gen0': '❌ Gen 0 — 매번 새 배치',
         'none': '—  sharpness 측정 없음'}

groups = {k: [] for k in LABEL}
for row in read_csv(os.path.join(PARSED, '_index.csv')):
    g = GEN.get(row['file'][:2], 'none')
    acc, sh = row.get('final_test_acc') or '', row.get('final_sharpness') or ''
    groups[g].append(dict(file=row['file'],
                          acc=float(acc) if acc else None,
                          sharp=float(sh) if sh else None))

usable = []
for g in ['gen2', 'vit', 'gen1', 'gen0', 'none']:
    if not groups[g]: continue
    print(f'{LABEL[g]}  ({len(groups[g])}개)')
    for it in groups[g]:
        a = f'{it["acc"]:.2f}%' if it['acc'] is not None else '   —  '
        s = f'{it["sharp"]:.4f}' if it['sharp'] is not None else '  —   '
        print(f'    acc {a:>8}  sharp {s:>8}  {it["file"]}')
        if g == 'gen2' and it['acc'] is not None and it['sharp'] is not None:
            usable.append(it)
    print()

# 같은 Stage 1 run 이 여러 노트북에 실려 중복 계수된다
seen, dedup = set(), []
for u in usable:
    k = (round(u['acc'],2), round(u['sharp'],4))
    if k in seen: print(f'  (중복 제외) {u["file"]}'); continue
    seen.add(k); dedup.append(u)

if len(dedup) >= 3:
    print(f'\nGen 2 상관 (중복 제거 n={len(dedup)}): '
          f'Pearson {pearson([u["sharp"] for u in dedup],[u["acc"] for u in dedup]):+.3f}')
print()
print('⚠️ 이 상관을 증거로 쓰면 안 된다.')
print('   남은 점들도 서로 독립이 아니다 — 각 노트북의 두 arm 은 같은 Stage 1')
print('   체크포인트에서 분기해 출발점을 공유한다. 유효 표본은 독립 run 3개 수준이고,')
print('   신뢰구간이 사실상 (-1, 1) 전체다. 주장은 위 Δ(차분 상관)에 싣는다.')
print()
print('⚠️ 베이스라인 3종(SGD 94.22 / AdamW 92.93 / SAM 94.69)에는 sharpness 가 없다.')
print('   고정확도 구간(92~95%)이 통째로 비어 있다는 뜻이다.')

In [ ]:
# ══ 5. 그림 ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, fn in DECAY_ARMS:
    rows = load(fn)
    if not rows: continue
    rows = [r for r in rows if r['sharpness'] and r['test_acc']]
    e = [float(r['epoch']) for r in rows]
    on = 'sam_on' in name
    style = '-' if on else '--'
    color = ('tab:blue' if on else 'tab:red') if name.startswith('ViT') \
            else ('tab:green' if on else 'tab:orange')
    axes[0].plot(e, [float(r['test_acc']) for r in rows], style,
                 color=color, marker='o', ms=3, label=name)
    axes[1].plot(e, [float(r['sharpness']) for r in rows], style,
                 color=color, marker='o', ms=3, label=name)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Accuracy rises during decay')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Sharpness (one-sided)')
axes[1].set_title('Sharpness does not follow')
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUTDIR}/fig1_decay_trajectories.png',
                                dpi=150, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
xs = range(len(traj))
ax.bar([x-.2 for x in xs], [t['raw'] for t in traj], .4,
       label='raw correlation', color='tab:gray')
ax.bar([x+.2 for x in xs], [t['detrended'] for t in traj], .4,
       label='detrended (first difference)', color='tab:purple')
ax.axhline(0, color='k', lw=.8)
ax.set_xticks(list(xs)); ax.set_xticklabels([t['arm'] for t in traj],
                                            rotation=20, ha='right', fontsize=8)
ax.set_ylabel('corr(sharpness, accuracy)')
ax.set_title('Raw correlation is a time-trend artifact')
ax.legend(fontsize=8); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'{OUTDIR}/fig2_detrended.png',
                                dpi=150, bbox_inches='tight'); plt.show()

json.dump({'trajectories': traj, 'gen2_scatter': dedup},
          open(f'{OUTDIR}/phase0_results.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)
print('저장:', OUTDIR)